<!--
SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
SPDX-License-Identifier: Apache-2.0
-->

# Aim
The purpose of this notebook is to demo AITune module wrapper.

In [ ]:
%load_ext autoreload
%autoreload 2
%cd ..
%pip install timm>=1.0.14

/home/pbazan/sources/ai-tune


In [2]:
import copy
import os
from logging import basicConfig

import torch

from aitune.global_context import BATCH_SIZE_KEY, global_context
from aitune.torch.backend.torch_inductor_backend import TorchInductorJitBackend
from aitune.torch.module.wrapper_module import Module
from aitune.torch.tune_strategy.one_backend_strategy import OneBackendStrategy

log_level = os.environ.get("AITUNE_LOG_LEVEL", "INFO")
basicConfig(level=log_level, format="%(asctime)s - %(levelname)s - %(message)s", force=True)

/home/pbazan/sources/ai-tune/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
TensorRT-LLM is not installed. Please install TensorRT-LLM or set TRTLLM_PLUGINS_PATH to the directory containing libnvinfer_plugin_tensorrt_llm.so to use converters for torch.distributed ops


[10/20/2025-17:04:31] [TRT] [W] Functionality provided through tensorrt.plugin module is experimental.


## Simple identity model
Let's start with something simple.

In [3]:
class Identity(torch.nn.Module):
    """Takes one input and returns it."""

    def forward(self, x, **kwargs):
        return x

#### Let's wrap module

In [4]:
model = Identity()
module = Module(model, "demo-identity")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#### Record some samples

If you call a wrapped module it will save sample input and output data of a model.

The following is an example how to call a model and record data.

In [5]:
module(torch.tensor(1), a=True)

tensor(1)

Apart from saving data (which is used later on during tuning) it will also create graph specifications.

A `graph spec` has a `name` and corresponding 
- `input_spec` - a way of describing model inputs: `args` and `kwargs`
- `output_spec` - a way of describing model outputs

Let's look at the `graph_spec`.

In [6]:
print(module.graph_specs[0])

Name=0
Input_spec:
Tensors:
╒═══════════╤════════╤═════════╤═════════════╤═════════════╤═════════════╕
│ Locator   │ Name   │ Shape   │ Min Shape   │ Max Shape   │ Dtype       │
╞═══════════╪════════╪═════════╪═════════════╪═════════════╪═════════════╡
│ [0]       │ args_0 │ []      │ []          │ []          │ torch.int64 │
╘═══════════╧════════╧═════════╧═════════════╧═════════════╧═════════════╛
Other:
╒═══════════╤══════════╤═════════╕
│ Locator   │ Name     │ Value   │
╞═══════════╪══════════╪═════════╡
│ ['a']     │ kwargs_a │ True    │
╘═══════════╧══════════╧═════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤═════════╤═════════════╤═════════════╤═════════════╕
│ Locator   │ Name    │ Shape   │ Min Shape   │ Max Shape   │ Dtype       │
╞═══════════╪═════════╪═════════╪═════════════╪═════════════╪═════════════╡
│           │ outputs │ []      │ []          │ []          │ torch.int64 │
╘═══════════╧═════════╧═════════╧═════════════╧═════════════╧═════════════╛



The `input_spec` show one tensor, as the first argument `args_0`. The `locator` is a recipe how to extract this tensor from recorded data.

Apart from the tensor we have also `other` data i.e. one keyword argument `kwargs_a`.

Based on the `input_spec` Ai Tune tries to create different `graph_specs` (i.e. the `output_spec` does not play role here).

Let's record another sample and see if the number of graphs changed.

In [7]:
print("Number of graphs before second sample", len(module.graph_specs))
module(torch.tensor(2), a=True)
print("Number of graphs after second sample", len(module.graph_specs))

Number of graphs before second sample 1
Number of graphs after second sample 1


Since the tensor rank did not change (ony its value) - there is no new graphs created. However if the other arguments change or the rank,
 there will be a new graph_spec craeted.

 Let's change the kwarg argument.

In [8]:
print("Number of graphs before second sample", len(module.graph_specs))
module(torch.tensor(2), a=False)
print("Number of graphs after second sample", len(module.graph_specs))

Number of graphs before second sample 1
Number of graphs after second sample 2


Let's record a sample with a different rank of a tensor.

In [9]:
module(torch.randn(1, 1))

# print all graph specs
for gs in module.graph_specs:
    print('-'*100)
    print(gs)

----------------------------------------------------------------------------------------------------
Name=0
Input_spec:
Tensors:
╒═══════════╤════════╤═════════╤═════════════╤═════════════╤═════════════╕
│ Locator   │ Name   │ Shape   │ Min Shape   │ Max Shape   │ Dtype       │
╞═══════════╪════════╪═════════╪═════════════╪═════════════╪═════════════╡
│ [0]       │ args_0 │ []      │ []          │ []          │ torch.int64 │
╘═══════════╧════════╧═════════╧═════════════╧═════════════╧═════════════╛
Other:
╒═══════════╤══════════╤═════════╕
│ Locator   │ Name     │ Value   │
╞═══════════╪══════════╪═════════╡
│ ['a']     │ kwargs_a │ True    │
╘═══════════╧══════════╧═════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤═════════╤═════════════╤═════════════╤═════════════╕
│ Locator   │ Name    │ Shape   │ Min Shape   │ Max Shape   │ Dtype       │
╞═══════════╪═════════╪═════════╪═════════════╪═════════════╪═════════════╡
│           │ outputs │ []      │ []          │ []          │ to

As can be seen, third graph has tensor as input with shapes `[1, 1]` and min and max shapes being the same.

If we record more samples of different shapes, the graph will have dynamic dimensions calculated.

In [10]:
module(torch.randn(2, 2))
module.graph_specs[-1]

Name=2
Input_spec:
Tensors:
╒═══════════╤════════╤══════════════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name   │ Shape            │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪════════╪══════════════════╪═════════════╪═════════════╪═══════════════╡
│ [0]       │ args_0 │ ['dim0', 'dim1'] │ [1, 1]      │ [2, 2]      │ torch.float32 │
╘═══════════╧════════╧══════════════════╧═════════════╧═════════════╧═══════════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤══════════════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name    │ Shape            │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪═════════╪══════════════════╪═════════════╪═════════════╪═══════════════╡
│           │ outputs │ ['dim0', 'dim1'] │ [1, 1]      │ [2, 2]      │ torch.float32 │
╘═══════════╧═════════╧══════════════════╧═════════════╧═════════════╧═══════════════╛

As can be seen, dynamic shapes are named `dim0`, `dim1` - they were calculated because we recorded same tensor with different shapes meaning they have to be dynamic.

The above also shows that graph has seen min and max shapes of `[1, 1]` and `[2, 2]` respectively i.e. first dimension is from 1 to 2, same for the second.

In [11]:
module.state

<ModuleState.RECORDING: 'recording'>

#### Automatic batch dimension detection

As we have seen module detects dynamic axes just by looking at different input data. However for checking performance and sanity AiTune needs to detect batch axis - which is special type of dynamic axis i.e. grows with a batch size. One example could be LLM pipeline where we have batch and sequence length dynamic axes `B` and `L` but only `B` grows with a batch size.

In order to detect batch dimension Module wrapper has to have information about current global batch size. This is because the `global batch size` can be different than `local batch size` e.g. SDXL model stacks input tensor and with `global bs=1` a module can receive `local bs=2`.

When you perform tuning, the AiTune takes dataloader and populates information about `batch size` so that module wrapper can obtain it. Since we are demonstrating module wrapper in isolation, we have to provide that information ourselves.

In [12]:
with global_context:
    global_context[BATCH_SIZE_KEY] = 1
    module(torch.randn(1, 2, 3))
    global_context[BATCH_SIZE_KEY] = 2
    module(torch.randn(2, 4, 3,))

In [13]:
module.graph_specs[-1]


Name=3
Input_spec:
Tensors:
╒═══════════╤════════╤═════════════════════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name   │ Shape                   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪════════╪═════════════════════════╪═════════════╪═════════════╪═══════════════╡
│ [0]       │ args_0 │ ['batch0', 'batch1', 3] │ [1, 2, 3]   │ [2, 4, 3]   │ torch.float32 │
╘═══════════╧════════╧═════════════════════════╧═════════════╧═════════════╧═══════════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤═════════════════════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name    │ Shape                   │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪═════════╪═════════════════════════╪═════════════╪═════════════╪═══════════════╡
│           │ outputs │ ['batch0', 'batch1', 3] │ [1, 2, 3]   │ [2, 4, 3]   │ torch.float32 │
╘═══════════╧═════════╧═════════════════════════╧═════════════╧═════════════╧═══════════════╛

As can be seen, the last graph has batch axis `batch0` which grows with batch size and `batch1` which grows twice the batch size.

#### Check changing state to passthrough and back to recording.

You can switch a wrappe module to a passthrough mode. Than samples are not recorded. Hence the graph should not be altered.

In [14]:
print("Number of graphs before second sample", len(module.graph_specs))
module(torch.randn(1, 2, 3))
print("Number of graphs after second sample", len(module.graph_specs))

Number of graphs before second sample 4
Number of graphs after second sample 4


#### Let's try tune dry-run

In [15]:
module.enable_recording()
module.tune(dry_run=True, device=device)

2025-10-20 17:04:32,024 - INFO - ------------------------------------------------------------
2025-10-20 17:04:32,024 - INFO - 🚀 Tuning graph `0` for module `demo-identity` (DRY RUN):
2025-10-20 17:04:32,024 - INFO -   number of parameters: 0
2025-10-20 17:04:32,024 - INFO -   number of layers: 0
2025-10-20 17:04:32,025 - INFO -   precisions: 
2025-10-20 17:04:32,025 - INFO -   graph_spec:
2025-10-20 17:04:32,025 - INFO -     input_spec:
 Tensors:
╒═══════════╤════════╤═════════╤═════════════╤═════════════╤═════════════╕
│ Locator   │ Name   │ Shape   │ Min Shape   │ Max Shape   │ Dtype       │
╞═══════════╪════════╪═════════╪═════════════╪═════════════╪═════════════╡
│ [0]       │ args_0 │ []      │ []          │ []          │ torch.int64 │
╘═══════════╧════════╧═════════╧═════════════╧═════════════╧═════════════╛
Other:
╒═══════════╤══════════╤═════════╕
│ Locator   │ Name     │ Value   │
╞═══════════╪══════════╪═════════╡
│ ['a']     │ kwargs_a │ True    │
╘═══════════╧══════════╧══

In [16]:
module.state

<ModuleState.RECORDING: 'recording'>

# Resnet50

In [17]:
# example from https://huggingface.co/docs/timm/en/models/resnet
import urllib
from pathlib import Path

import timm
from PIL import Image
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform


In [18]:
model = timm.create_model('resnet50', pretrained=True)
model.to('cuda')
model.eval()
config = resolve_data_config({}, model=model)
transform = create_transform(**config)

url, filename = ("https://github.com/pytorch/hub/raw/master/images/dog.jpg", "dog.jpg")
if not Path(filename).exists():
    urllib.request.urlretrieve(url, filename)

img = Image.open(filename).convert('RGB')
data = transform(img).unsqueeze(0).to('cuda')  # transform and add batch dimension
data.shape

2025-10-20 17:04:32,275 - INFO - Loading pretrained weights from Hugging Face hub (timm/resnet50.a1_in1k)
2025-10-20 17:04:33,064 - INFO - [timm/resnet50.a1_in1k] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.


torch.Size([1, 3, 224, 224])

In [19]:
with torch.no_grad():
    out = model(data)
ref_probs = torch.nn.functional.softmax(out[0], dim=0)


#### Let's wrap module

In [20]:
module = Module(model, "demo-resnet50")

#### Record some samples

In [21]:
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 1
    _ = module(data)

module.graph_specs[0]

Name=0
Input_spec:
Tensors:
╒═══════════╤════════╤══════════════════╤══════════════════╤══════════════════╤═══════════════╕
│ Locator   │ Name   │ Shape            │ Min Shape        │ Max Shape        │ Dtype         │
╞═══════════╪════════╪══════════════════╪══════════════════╪══════════════════╪═══════════════╡
│ [0]       │ args_0 │ [1, 3, 224, 224] │ [1, 3, 224, 224] │ [1, 3, 224, 224] │ torch.float32 │
╘═══════════╧════════╧══════════════════╧══════════════════╧══════════════════╧═══════════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤═══════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name    │ Shape     │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪═════════╪═══════════╪═════════════╪═════════════╪═══════════════╡
│           │ outputs │ [1, 1000] │ [1, 1000]   │ [1, 1000]   │ torch.float32 │
╘═══════════╧═════════╧═══════════╧═════════════╧═════════════╧═══════════════╛

In [22]:
# Let's record bs = 2 to have dynamic batch dimension

In [23]:
with global_context as ctx:
    ctx[BATCH_SIZE_KEY] = 2
    _ = module(data.repeat(2, 1, 1, 1))

In [24]:
module.graph_specs[0]

Name=0
Input_spec:
Tensors:
╒═══════════╤════════╤═════════════════════════╤══════════════════╤══════════════════╤═══════════════╕
│ Locator   │ Name   │ Shape                   │ Min Shape        │ Max Shape        │ Dtype         │
╞═══════════╪════════╪═════════════════════════╪══════════════════╪══════════════════╪═══════════════╡
│ [0]       │ args_0 │ ['batch0', 3, 224, 224] │ [1, 3, 224, 224] │ [2, 3, 224, 224] │ torch.float32 │
╘═══════════╧════════╧═════════════════════════╧══════════════════╧══════════════════╧═══════════════╛
Output_spec:
Tensors:
╒═══════════╤═════════╤══════════════════╤═════════════╤═════════════╤═══════════════╕
│ Locator   │ Name    │ Shape            │ Min Shape   │ Max Shape   │ Dtype         │
╞═══════════╪═════════╪══════════════════╪═════════════╪═════════════╪═══════════════╡
│           │ outputs │ ['batch0', 1000] │ [1, 1000]   │ [2, 1000]   │ torch.float32 │
╘═══════════╧═════════╧══════════════════╧═════════════╧═════════════╧═══════════════╛

#### Let's try tune dry-run

In [25]:
torch_compile_strategy = OneBackendStrategy(TorchInductorJitBackend())

In [26]:
module.tune(strategy=torch_compile_strategy, dry_run=True, device=device)

2025-10-20 17:04:33,561 - INFO - ------------------------------------------------------------
2025-10-20 17:04:33,561 - INFO - 🚀 Tuning graph `0` for module `demo-resnet50` (DRY RUN):
2025-10-20 17:04:33,562 - INFO -   number of parameters: 25.6M
2025-10-20 17:04:33,562 - INFO -   number of layers: 10
2025-10-20 17:04:33,562 - INFO -   precisions: torch.float32
2025-10-20 17:04:33,563 - INFO -   graph_spec:
2025-10-20 17:04:33,563 - INFO -     input_spec:
 Tensors:
╒═══════════╤════════╤═════════════════════════╤══════════════════╤══════════════════╤═══════════════╕
│ Locator   │ Name   │ Shape                   │ Min Shape        │ Max Shape        │ Dtype         │
╞═══════════╪════════╪═════════════════════════╪══════════════════╪══════════════════╪═══════════════╡
│ [0]       │ args_0 │ ['batch0', 3, 224, 224] │ [1, 3, 224, 224] │ [2, 3, 224, 224] │ torch.float32 │
╘═══════════╧════════╧═════════════════════════╧══════════════════╧══════════════════╧═══════════════╛

2025-10-20 17:

#### Let's try real tune with torch compile

In [27]:
module.tune(strategy=torch_compile_strategy, dry_run=False, device=device)

2025-10-20 17:04:33,591 - INFO - 🚀 Finding max batch size for demo-resnet50
2025-10-20 17:04:34,402 - INFO - ✅ Max batch size for demo-resnet50 is 16 with throughput 7059.08 samples/s
2025-10-20 17:04:34,404 - INFO - ------------------------------------------------------------
2025-10-20 17:04:34,404 - INFO - 🚀 Tuning graph `0` for module `demo-resnet50`:
2025-10-20 17:04:34,405 - INFO -   number of parameters: 25.6M
2025-10-20 17:04:34,405 - INFO -   number of layers: 10
2025-10-20 17:04:34,406 - INFO -   precisions: torch.float32
2025-10-20 17:04:34,406 - INFO -   graph_spec:
2025-10-20 17:04:34,406 - INFO -     input_spec:
 Tensors:
╒═══════════╤════════╤═════════════════════════╤══════════════════╤═══════════════════╤═══════════════╕
│ Locator   │ Name   │ Shape                   │ Min Shape        │ Max Shape         │ Dtype         │
╞═══════════╪════════╪═════════════════════════╪══════════════════╪═══════════════════╪═══════════════╡
│ [0]       │ args_0 │ ['batch0', 3, 224, 22

In [28]:
out = module(data)
actual_probs = torch.nn.functional.softmax(out[0], dim=0)

In [29]:
module.state

<ModuleState.TUNED: 'tuned'>

In [30]:
torch.testing.assert_close(ref_probs, actual_probs, rtol=1e-4, atol=1e-4)